# Hydraulic System Anomaly Detection - Machine Learning Pipeline
This notebook implements an end-to-end classification pipeline to detect operational sensor anomalies (`anomaly_label`) for the **Hydraulic System** component without using `failure_within_50_hours` or `sensor_status` as features.

In [1]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Dataset (`hydraulic_system.csv`)
Loading the telemetry training dataset for `hydraulic_system`.

In [2]:
try:
    df = pd.read_csv("../Data/hydraulic_system.csv")
except Exception:
    try:
        df = pd.read_csv("Data/hydraulic_system.csv")
    except Exception:
        try:
            df = pd.read_csv("src/ML/Data/hydraulic_system.csv")
        except Exception:
            df = pd.read_csv("C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/Data/hydraulic_system.csv")

print("Hydraulic System dataset successfully loaded into pandas DataFrame.")
print("=" * 60)
print("DATASET SHAPE:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("=" * 60)


Hydraulic System dataset successfully loaded into pandas DataFrame.
DATASET SHAPE:
Rows: 4889, Columns: 18


## 3. Comprehensive Dataset Overview
Displaying dataset column names, data types, missing values, and target class distribution (`anomaly_label`).

In [3]:
print("COLUMN NAMES:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION OF TARGET (anomaly_label):")
print(df["anomaly_label"].value_counts())


CLASS DISTRIBUTION OF TARGET (anomaly_label):
anomaly_label
0    3167
1    1722
Name: count, dtype: int64


## 4. Preprocessing Pipeline & Feature Selection
- Target: `anomaly_label`
- **Explicit Constraints**: Neither `failure_within_50_hours` nor `sensor_status` is used as a feature.
- Identifiers dropped: `asset_id`, `timestamp`, `component_id`, `component_type`.
- Features used: Only raw sensor and operational telemetry.
- Numerical pipeline: `SimpleImputer(strategy="median")` followed by `StandardScaler()`.

In [4]:
target = "anomaly_label"
# Exclude identifiers, failure_within_50_hours, and sensor_status
drop_cols = [
    "asset_id", "timestamp", "component_id", "component_type",
    "failure_within_50_hours", "sensor_status", target
]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[target]

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Selected Numerical Features ({len(num_cols)}):\n{num_cols}\n")
print(f"Selected Categorical Features ({len(cat_cols)}):\n{cat_cols}\n")

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
transformers = [("num", num_transformer, num_cols)]
if len(cat_cols) > 0:
    transformers.append(("cat", cat_transformer, cat_cols))

preprocessor = ColumnTransformer(
    transformers=transformers,
    verbose_feature_names_out=False,
)
print("Preprocessing pipeline configured (excluding failure_within_50_hours and sensor_status).")

Selected Numerical Features (11):
['temperature', 'vibration', 'oil_pressure', 'fuel_pressure', 'rpm', 'hydraulic_pressure', 'battery_voltage', 'coolant_temperature', 'operating_hours', 'load_percentage', 'ambient_temperature']

Selected Categorical Features (0):
[]

Preprocessing pipeline configured (excluding failure_within_50_hours and sensor_status).


## 5. Stratified Train-Test Split (80/20)
Splitting the telemetry dataset using `stratify=y` to preserve exact anomaly proportions.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("=" * 60)
print("DATASET SPLIT SUMMARY:")
print(f"  Total samples:  {len(df)}")
print(f"  X_train samples: {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"  X_test samples:  {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
print("=" * 60)

DATASET SPLIT SUMMARY:
  Total samples:  4889
  X_train samples: 3911 (80.0%)
  X_test samples:  978 (20.0%)


## 6. Model Training with Optimized Random Forest Classifier
Training an ensemble classifier with balanced weighting to accurately detect operational telemetry anomalies without ground-truth leakage.

In [6]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=16,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )),
])
pipeline.fit(X_train, y_train)

final_feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Model training complete.")
print(f"\nFinal feature names used by the model ({len(final_feature_names)}):\n{list(final_feature_names)}")

Model training complete.

Final feature names used by the model (11):
['temperature', 'vibration', 'oil_pressure', 'fuel_pressure', 'rpm', 'hydraulic_pressure', 'battery_voltage', 'coolant_temperature', 'operating_hours', 'load_percentage', 'ambient_temperature']


## 7. Feature Importance Analysis
Ranking physical sensor features by Gini importance for anomaly detection.

In [7]:
importances = pipeline.named_steps["classifier"].feature_importances_
fi_df = pd.DataFrame({"Feature": final_feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False).reset_index(drop=True)
print("=" * 60)
print("FEATURE IMPORTANCES (Highest to Lowest):")
print("=" * 60)
print(fi_df.to_string(index=False))

FEATURE IMPORTANCES (Highest to Lowest):
            Feature  Importance
          vibration    0.279625
        temperature    0.229763
       oil_pressure    0.166585
    load_percentage    0.110242
coolant_temperature    0.095766
    operating_hours    0.038499
    battery_voltage    0.032069
 hydraulic_pressure    0.017539
      fuel_pressure    0.011976
                rpm    0.009037
ambient_temperature    0.008899


## 8. Model Evaluation on Training and Testing Sets
Evaluating accuracy, precision, recall, F1, and ROC-AUC score.

In [8]:
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)
y_test_prob = pipeline.predict_proba(X_test)[:, 1]

print("=" * 60)
print("MODEL EVALUATION METRICS SUMMARY:")
print("=" * 60)
print(f"  Training Accuracy:  {accuracy_score(y_train, y_train_pred):.4f} ({accuracy_score(y_train, y_train_pred)*100:.2f}%)")
print(f"  Testing Accuracy:   {accuracy_score(y_test, y_test_pred):.4f} ({accuracy_score(y_test, y_test_pred)*100:.2f}%)")
print(f"  Precision:          {precision_score(y_test, y_test_pred):.4f} ({precision_score(y_test, y_test_pred)*100:.2f}%)")
print(f"  Recall:             {recall_score(y_test, y_test_pred):.4f} ({recall_score(y_test, y_test_pred)*100:.2f}%)")
print(f"  F1 Score:           {f1_score(y_test, y_test_pred):.4f}")
print(f"  ROC-AUC Score:      {roc_auc_score(y_test, y_test_prob):.4f}")
print("=" * 60)

MODEL EVALUATION METRICS SUMMARY:
  Training Accuracy:  0.9990 (99.90%)
  Testing Accuracy:   0.9632 (96.32%)
  Precision:          0.9503 (95.03%)
  Recall:             0.9448 (94.48%)
  F1 Score:           0.9475
  ROC-AUC Score:      0.9951


## 9. Example Prediction on Raw Telemetry Reading

In [9]:
example_sample = X_test.iloc[[0]]
example_pred = pipeline.predict(example_sample)[0]
example_prob = pipeline.predict_proba(example_sample)[0]

print("=" * 60)
print("EXAMPLE PREDICTION ON RAW SENSOR INPUT:")
print("=" * 60)
print(f"Predicted Class: {example_pred} ('{'Anomaly' if example_pred == 1 else 'Normal'}')")
print(f"Normal Probability:  {example_prob[0]:.4f} ({example_prob[0]*100:.2f}%)")
print(f"Anomaly Probability: {example_prob[1]:.4f} ({example_prob[1]*100:.2f}%)")
print("=" * 60)

EXAMPLE PREDICTION ON RAW SENSOR INPUT:
Predicted Class: 1 ('Anomaly')
Normal Probability:  0.0101 (1.01%)
Anomaly Probability: 0.9899 (98.99%)


## 10. Save Complete Pipeline to `src/ML/model/hydraulic_system_anomaly_model.pkl`

In [10]:
model_dir = None
for candidate in [
    Path("../model"),
    Path("model"),
    Path("src/ML/model"),
    Path("C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/model"),
]:
    if candidate.exists():
        model_dir = candidate
        break
if model_dir is None:
    model_dir = Path("C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/model")
    model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "hydraulic_system_anomaly_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(pipeline, f)

print(f"Model saved successfully to:\n  {model_path}")
with open(model_path, "rb") as f:
    loaded_pipeline = pickle.load(f)
print("Loaded Model Verification from Pickle:")
print(f"  Verified Predicted Class:      {loaded_pipeline.predict(example_sample)[0]}")
print(f"  Verified Anomaly Probability:  {loaded_pipeline.predict_proba(example_sample)[0][1]:.4f}")

Model saved successfully to:
  ..\model\hydraulic_system_anomaly_model.pkl
Loaded Model Verification from Pickle:
  Verified Predicted Class:      1
  Verified Anomaly Probability:  0.9899


## 11. Evaluation Matrix & Diagnostic Report
Confusion matrix, normalized counts, extended diagnostic statistics, and full classification report.

In [11]:
cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(cm, index=["Actual Normal (0)", "Actual Anomaly (1)"], columns=["Predicted Normal (0)", "Predicted Anomaly (1)"])
cm_norm = (confusion_matrix(y_test, y_test_pred, normalize="true") * 100).round(2)
cm_norm_df = pd.DataFrame(cm_norm, index=["Actual Normal (0)", "Actual Anomaly (1)"], columns=["Predicted Normal (%)", "Predicted Anomaly (%)"])

print("=" * 70)
print("                     FINAL EVALUATION MATRIX")
print("=" * 70)
print("\n1. CONFUSION MATRIX (Sample Counts):\n" + "-" * 50)
print(cm_df)
print("\n2. CONFUSION MATRIX (Normalized Class-wise %):\n" + "-" * 50)
print(cm_norm_df)
print("\n3. FULL CLASSIFICATION REPORT:\n" + "-" * 70)
print(classification_report(y_test, y_test_pred, target_names=["Normal (0)", "Anomaly (1)"], digits=4))
print("=" * 70)

                     FINAL EVALUATION MATRIX

1. CONFUSION MATRIX (Sample Counts):
--------------------------------------------------
                    Predicted Normal (0)  Predicted Anomaly (1)
Actual Normal (0)                    617                     17
Actual Anomaly (1)                    19                    325

2. CONFUSION MATRIX (Normalized Class-wise %):
--------------------------------------------------
                    Predicted Normal (%)  Predicted Anomaly (%)
Actual Normal (0)                  97.32                   2.68
Actual Anomaly (1)                  5.52                  94.48

3. EXTENDED DIAGNOSTIC METRICS:
--------------------------------------------------
  Accuracy:             0.9632 (96.32%)
  Sensitivity (Recall): 0.9448 (94.48%)
  Specificity:          0.9732 (97.32%)
  Precision (PPV):      0.9503 (95.03%)
  Negative Pred Value:  0.9701 (97.01%)
  F1 Score:             0.9475
  ROC AUC Score:        0.9951
  Matthews Corr Coef:   0.9192

4. FU

## 12. Inference on Test Dataset (`hydraulic_system_test.csv`)
Loading `hydraulic_system_test.csv`, extracting physical sensor features (neither `failure_within_50_hours` nor `sensor_status` used), computing anomaly probability percentages (`anomaly_probability_percent` and `anomalies_percentage`), applying the 40% threshold to flag anomalies (`anomaly_label`), and saving back to disk.

In [12]:
# 1. Locate and load test dataset
test_data_path = None
for candidate in [
    Path("../Data/hydraulic_system_test.csv"),
    Path("Data/hydraulic_system_test.csv"),
    Path("src/ML/Data/hydraulic_system_test.csv"),
    Path("C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/Data/hydraulic_system_test.csv"),
]:
    if candidate.exists():
        test_data_path = candidate
        break

df_test = pd.read_csv(test_data_path)
print("=" * 70)
print(f"Loaded Test Dataset from: {test_data_path}")
print(f"Initial Shape: {df_test.shape[0]} rows, {df_test.shape[1]} columns")
print("=" * 70)

# 2. Extract feature columns (DO NOT use failure_within_50_hours or sensor_status)
drop_test_cols = [
    "asset_id", "timestamp", "component_id", "component_type",
    "failure_within_50_hours", "failure_probability_percent",
    "sensor_status",
    "anomaly_label", "anomaly_probability_percent", "anomalies_percentage",
]
X_test_input = df_test.drop(columns=[c for c in drop_test_cols if c in df_test.columns])
print("\nFeatures passed into pipeline for anomaly detection:")
print(X_test_input.columns.tolist())

# 3. Predict anomaly probability percentage using pipeline
anomaly_prob = pipeline.predict_proba(X_test_input)[:, 1] * 100

# 4. Add probability percentage columns (both naming conventions supported)
df_test["anomaly_probability_percent"] = anomaly_prob.round(2)
df_test["anomalies_percentage"] = df_test["anomaly_probability_percent"]

# 5. Apply 40% threshold: if probability > 40%, flag as anomaly (1), else normal (0)
df_test["anomaly_label"] = (df_test["anomaly_probability_percent"] > 40.0).astype(int)

# 6. Save updated test data to CSV
df_test.to_csv(test_data_path, index=False)
print(f"\n[SUCCESS] Successfully written {len(df_test)} rows to '{test_data_path}'")
print(f"Final Shape: {df_test.shape[0]} rows, {df_test.shape[1]} columns")

# 7. Verification Assertions
assert set(df_test["anomaly_label"].unique()).issubset({0, 1}), "Invalid anomaly_label values"
assert (df_test["anomaly_probability_percent"] >= 0).all() and (df_test["anomaly_probability_percent"] <= 100).all(), "Invalid probability range"
assert df_test.isnull().sum().sum() == 0, "Missing/null values detected"
assert len(df_test) == 978, f"Expected 978 rows, got {len(df_test)}"

print("\n" + "=" * 70)
print("TEST SET ANOMALY PREDICTION SUMMARY (Threshold: > 40% -> Anomaly)")
print("=" * 70)
print(f"- Predicted Anomalies (>40%): {sum(df_test['anomaly_label'] == 1)} ({sum(df_test['anomaly_label'] == 1)/len(df_test)*100:.2f}%)")
print(f"- Predicted Normal (<=40%):   {sum(df_test['anomaly_label'] == 0)} ({sum(df_test['anomaly_label'] == 0)/len(df_test)*100:.2f}%)")
print(f"- Mean Anomaly Probability:   {df_test['anomaly_probability_percent'].mean():.2f}%")

# 8. Preview of Columns
preview_cols = [
    c for c in [
        "asset_id",
        "timestamp",
        "component_id",
        "anomaly_label",
        "anomaly_probability_percent",
        "anomalies_percentage",
        "failure_within_50_hours",
        "failure_probability_percent",
    ] if c in df_test.columns
]
print("\n" + "=" * 70)
print("REQUIRED COLUMNS PREVIEW (First 10 Rows):")
print("=" * 70)
print(df_test[preview_cols].head(10))


Loaded Test Dataset from: ..\Data\hydraulic_system_test.csv
Initial Shape: 978 rows, 21 columns

[SUCCESS] Successfully written 978 rows to '..\Data\hydraulic_system_test.csv'
Final Shape: 978 rows, 21 columns

TEST SET ANOMALY PREDICTION SUMMARY (Threshold: > 40% -> Anomaly)
Predicted Anomalies (>40%): 900 (92.02%)
Predicted Normal (<=40%):   78 (7.98%)
Mean Anomaly Probability:   89.31%

REQUIRED COLUMNS PREVIEW (First 10 Rows):
  asset_id            timestamp component_id  anomaly_label  anomaly_probability_percent  anomalies_percentage  failure_within_50_hours  failure_probability_percent
0     A025  2025-01-14 10:00:00     A025-HYD              1                        94.96                 94.96                        1                        75.20
1     A028  2025-01-14 10:00:00     A028-HYD              1                        99.01                 99.01                        0                        33.20
2     A036  2025-01-14 10:00:00     A036-HYD              0           